# 03 · Temperatura mínima y máxima con la API de FluvioTech

Este notebook consulta temperatura diaria PISCO para:

- polígono con promedio espacial;
- polígono con extracción por píxel;
- cálculo de temperatura media diaria;
- gráficos básicos.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

# Cambia a la URL pública si no estás trabajando en modo local.
URL_TEMP = "https://fluviotech.com/meteodata/api/pisco/temperature/daily/"
# URL_TEMP = "http://localhost:8000/meteodata/api/pisco/temperature/daily/"

In [ ]:
def consultar_temperatura(payload):
    """Consulta la API de temperatura diaria y devuelve un DataFrame."""
    response = requests.post(URL_TEMP, json=payload)

    print("Código de estado:", response.status_code)
    print("Vista previa:", response.text[:300])

    response.raise_for_status()
    result = response.json()

    df = pd.DataFrame(result["data"])
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])

    return result, df

## 1. Polígono de ejemplo

In [ ]:
poligono_lima = {
    "type": "Polygon",
    "coordinates": [[
        [-77.20, -12.20],
        [-76.90, -12.20],
        [-76.90, -11.90],
        [-77.20, -11.90],
        [-77.20, -12.20]
    ]]
}

## 2. Temperatura promedio del polígono

In [ ]:
payload_mean = {
    "geometry": poligono_lima,
    "start": "2020-01-01",
    "end": "2020-01-10",
    "variables": ["tmin", "tmax"],
    "aggregation": "mean"
}

result_mean, df_temp_mean = consultar_temperatura(payload_mean)

print("Columnas:", df_temp_mean.columns.tolist())
df_temp_mean.head()

In [ ]:
# Creamos temperatura media si existen tmin y tmax
if {"tmin", "tmax"}.issubset(df_temp_mean.columns):
    df_temp_mean["tmean"] = (df_temp_mean["tmin"] + df_temp_mean["tmax"]) / 2

df_temp_mean.head()

In [ ]:
plt.figure(figsize=(9, 4))

if "tmin" in df_temp_mean.columns:
    plt.plot(df_temp_mean["date"], df_temp_mean["tmin"], marker="o", label="Tmin")

if "tmax" in df_temp_mean.columns:
    plt.plot(df_temp_mean["date"], df_temp_mean["tmax"], marker="o", label="Tmax")

if "tmean" in df_temp_mean.columns:
    plt.plot(df_temp_mean["date"], df_temp_mean["tmean"], marker="o", label="Tmean")

plt.title("Temperatura diaria promedio del polígono")
plt.xlabel("Fecha")
plt.ylabel("Temperatura (°C)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 3. Temperatura por píxel

In [ ]:
payload_pixels = {
    "geometry": poligono_lima,
    "start": "2020-01-01",
    "end": "2020-01-10",
    "variables": ["tmin", "tmax"],
    "aggregation": "pixels"
}

result_pixels, df_temp_pixels = consultar_temperatura(payload_pixels)

print("Columnas:", df_temp_pixels.columns.tolist())
df_temp_pixels.head()

In [ ]:
# Temperatura media por píxel, si las columnas están disponibles
if {"tmin", "tmax"}.issubset(df_temp_pixels.columns):
    df_temp_pixels["tmean"] = (df_temp_pixels["tmin"] + df_temp_pixels["tmax"]) / 2

df_temp_pixels.head()

In [ ]:
# Graficar algunos píxeles para Tmax
if "pixel_id" in df_temp_pixels.columns and "tmax" in df_temp_pixels.columns:
    pixeles = df_temp_pixels["pixel_id"].drop_duplicates().head(5)

    plt.figure(figsize=(10, 4))
    for pixel in pixeles:
        tmp = df_temp_pixels[df_temp_pixels["pixel_id"] == pixel]
        plt.plot(tmp["date"], tmp["tmax"], marker="o", label=f"pixel {pixel}")

    plt.title("Tmax diaria por píxel")
    plt.xlabel("Fecha")
    plt.ylabel("Temperatura máxima (°C)")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()
else:
    print("No se encontró pixel_id o tmax. Revisa las columnas disponibles.")

## Ejercicio

1. Cambia el polígono a otra zona.
2. Consulta un mes completo.
3. Calcula Tmax media del periodo por píxel.
4. Identifica el píxel con mayor Tmax promedio.

In [ ]:
if "pixel_id" in df_temp_pixels.columns and "tmax" in df_temp_pixels.columns:
    resumen_tmax_pixel = (
        df_temp_pixels
        .groupby("pixel_id", as_index=False)["tmax"]
        .mean()
        .rename(columns={"tmax": "tmax_promedio"})
        .sort_values("tmax_promedio", ascending=False)
    )

    resumen_tmax_pixel.head()